In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score , r2_score
import joblib
import streamlit as st
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
df=pd.read_csv('patients_pro_max_with_priority.csv')

In [ ]:
df.head()

In [3]:
# pain map = 3: severe , 2: moderate , 1:mild , 0:none
# condition_map = {
#     'Heart Attack': 1, 'Stroke': 2, 'Dislocated Knee': 3, 'Kidney Stones': 4,
#     'Appendicitis': 5, 'Torn ACL': 6, 'Bowel Obstruction': 7, 'Fracture': 8,
#     'Gallstones': 9, 'Hernia': 10
# }


In [ ]:
df['pain_level'].unique()

In [ ]:
df['condition'].unique()

In [ ]:
df['gender'].unique()

In [ ]:
df['gender'] = df['gender'].replace({'Female': 0, 'Male': 1})

In [ ]:
df['bp_mean'].isnull().sum()

In [ ]:
df.head()

In [ ]:
x=df[['age','gender','heart_rate','respiratory_rate','oxygen_saturation','pain_level','condition','bp_mean']]
y=df['needs_surgery']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print('training feature shape',x_train.shape)
print('testing feature shape',x_test.shape)

### Logistic Regression

In [4]:

print(x.isnull().values.any())
print("Total NaN values:", x.isnull().sum().sum())
print(x.isnull().sum())


NameError: name 'x' is not defined

In [1]:
param_dist_logreg = {
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'C': np.logspace(-3, 3, 10), 
    'solver': ['saga', 'liblinear'],
    'max_iter': [500, 1000, 2000]
}
log_reg = LogisticRegression()
random_search_logreg = RandomizedSearchCV(
    log_reg,
    param_distributions=param_dist_logreg,
    n_iter=20,   # try 20 random combinations
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search_logreg.fit(x_train, y_train)
print("Best Logistic Regression Params:", random_search_logreg.best_params_)
print("best scored  :",random_search_logreg.best_score_)

NameError: name 'np' is not defined

In [ ]:
log_reg = LogisticRegression(solver='saga',penalty=None,max_iter=500,C=np.float64(0.001))

# Train the model
log_reg.fit(x_train, y_train)

# Make predictions
y_pred = log_reg.predict(x_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Decision Tree Classifier

In [112]:
param_dist_tree = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

dtree = DecisionTreeClassifier()

random_search_tree = RandomizedSearchCV(
    dtree,
    param_distributions=param_dist_tree,
    n_iter=20,
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search_tree.fit(x_train, y_train)
print("Best Decision Tree Params:", random_search_tree.best_params_)
print("best scored  :",random_search_tree.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Decision Tree Params: {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 5, 'criterion': 'entropy'}
best scored  : 0.8254166666666666


In [ ]:
dtree = DecisionTreeClassifier(min_samples_split=10,min_samples_leaf=4,max_features='log2',max_depth=5,criterion='entropy')

# Train the model
dtree.fit(x_train, y_train)

# Make predictions
dtree = dtree.predict(x_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [17]:
y_pred_log = random_search_logreg.predict(x_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

# Decision Tree results
y_pred_tree = random_search_tree.predict(x_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

NameError: name 'random_search_logreg' is not defined

# RANDOM FOREST CLASSIFIER

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)
print("accuracy score:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [159]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

param_grid = {
    'n_estimators': [200, 400, 600, 800, 1000],
    'max_depth': [10, 20, 30, None],
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf_search = RandomizedSearchCV(
    rf, param_grid, n_iter=20, cv=5,
    scoring='f1_weighted', n_jobs=-1, random_state=42, verbose=2
)
rf_search.fit(x_train, y_train)

best_rf = rf_search.best_estimator_
print("Best RF params:", rf_search.best_params_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best RF params: {'n_estimators': 800, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': False}


In [ ]:
model = RandomForestClassifier(min_samples_leaf=1,max_depth=10, min_samples_split= 5,max_features='sqrt',bootstrap=False, n_estimators=800,random_state=42)

In [ ]:
model.fit(x_train,y_train)
model.score(x_train,y_train)

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, cmap='Blues',cbar=False)
plt.xlabel('Predicted Values')
plt.ylabel('Actual Values')
plt.title('Confusion Matrix for Random Forest Classifier')
plt.show()

In [ ]:
ax = sns.distplot(y_test, color='red',  label='Actual Value',hist=False)
sns.distplot(y_pred, color='blue', label='Predicted Value',hist=False,ax=ax)
plt.title('Actual vs Predicted Value Random Forest Classifier')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.show()

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [26]:
from sklearn.metrics import classification_report
print('the Random forest classifier accuracy is :',accuracy_score(y_test,y_pred))
print(classification_report(y_test, y_pred))

the Random forest classifier accuracy is : 0.8293333333333334
              precision    recall  f1-score   support

       False       0.73      0.37      0.49       669
        True       0.84      0.96      0.90      2331

    accuracy                           0.83      3000
   macro avg       0.79      0.67      0.69      3000
weighted avg       0.82      0.83      0.81      3000



In [2]:
# saving the model
joblib.dump(model, "need_surgery_rf.pkl")

NameError: name 'joblib' is not defined

# RANDOM FOREST REGRESSOR

In [27]:
param_grid_rfr= {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
rf=RandomForestRegressor(random_state=42)

In [30]:
grid_search_rfr= GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rfr,
    cv=5,
    n_jobs=-1,
    scoring='r2',
)
grid_search_rfr.fit(x_train,y_train)


print ("best parameters :",grid_search_rfr.best_params_)
print("best scored (CV) :",grid_search_rfr.best_score_)

best parameters : {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
best scored (CV) : 0.27595195523898935


In [28]:
best_rf = grid_search_rfr.best_estimator_
y_pred = best_rf.predict(x_test)

print("Test R²:", r2_score(y_test, y_pred))
print("Test RMSE:", mean_squared_error(y_test, y_pred, squared=False))


Test R²: 0.24081489031616543


TypeError: got an unexpected keyword argument 'squared'

### KNeighbors Classifi

In [133]:

knn=KNeighborsClassifier(n_neighbors=6)
knn.fit(x_train,y_train)

KNeighborsClassifier(n_neighbors=6)

In [134]:
y_pred=knn.predict(x_test)

In [132]:
from sklearn.model_selection import GridSearchCV
param_grid = {'n_neighbors': list(range(1, 21))}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, n_jobs=-1)
grid.fit(x_train, y_train)
print("Best k:", grid.best_params_)
print("Best cross-validated score:", grid.best_score_)
best_k = grid.best_params_['n_neighbors']
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(x_train, y_train)


Best k: {'n_neighbors': 20}
Best cross-validated score: 0.8201666666666666


KNeighborsClassifier(n_neighbors=20)